# CellTag3 analysis

In [ ]:
import json
from IPython.display import display, Markdown
import ipynbname

nb_path = ipynbname.path()
with open(nb_path) as f:
    nb = json.load(f)

all_headings = []
for cell in nb['cells']:
    if cell['cell_type'] == 'markdown':
        source = ''.join(cell['source'])
        if 'Table of Contents' in source:
            continue
        for line in source.split('\n'):
            line = line.strip()
            if line.startswith('#'):
                level = len(line) - len(line.lstrip('#'))
                title = line.lstrip('#').strip()
                all_headings.append((level, title))

min_level = min(h[0] for h in all_headings)
toc_lines = ["## Table of Contents\n Open outline or Ctrl+F to find\n"]
for level, title in all_headings:
    indent = '  ' * (level - min_level)
    toc_lines.append(f"{indent}- {title}")  # no link, just text

display(Markdown('\n'.join(toc_lines)))

In [1]:
import scclr

In [2]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
# ============================================================
# Fonts / style
# ============================================================
repo_path = "/home/gzu5140/Keerthana_b1042/TwINFER/code/TwINFER"
font_paths = [
    f"{repo_path}/fonts/Arial.ttf",
    f"{repo_path}/fonts/Arial Bold.ttf",
    f"{repo_path}/fonts/Arial Italic.ttf",
    f"{repo_path}/fonts/Arial Bold Italic.ttf",
]

for fp in font_paths:
    try:
        fm.fontManager.addfont(fp)
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

# ==== LaTeX + SVG text mode (Illustrator-safe) ====
plt.rcParams['pdf.fonttype'] = 42  # For PDF export
plt.rcParams['ps.fonttype'] = 42   # For PostScript (EPS) export
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 18     # x/y labels
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['xtick.labelsize'] = 12     # x-axis tick labels
plt.rcParams['ytick.labelsize'] = 12    # y-axis tick labels
plt.rcParams['legend.fontsize'] = 12    # legend text

## Import packages

In [ ]:
import scanpy as sc
import os
import pandas as pd
from singletCode import check_sample_sheet, get_singlets
import harmonypy
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import median_abs_deviation
from scipy.stats import mannwhitneyu

In [6]:
barcode = pd.read_csv("/home/gzu5140/Keerthana_b1042/TwINFER/real_data/cellTag3_data/all_samples.csv")
h5_dir = "/home/gzu5140/Keerthana_b1042/TwINFER/real_data/celltag_3_geo_data/"
output_path = "/home/gzu5140/Keerthana_b1042/analysis_data/celltag_3/"

In [ ]:
# sample_list = ['d5-RNA-2', 'd2-RNA-5', 'd5-RNA-1', 'd2-ATAC-5', 'd5-ATAC-1', 'd5-ATAC-2', 'd5-ATAC-3', 'd5-ATAC-4']
# barcode = barcode[barcode['sample'].isin(sample_list)]
# singlet_cells = get_singlets(barcode)
# singlet_cells_list = singlet_cells[0][singlet_cells[0]['label'] == "Singlet"]
# singlet_cells_list.to_csv("/home/gzu5140/Keerthana_b1042/TwINFER/real_data/cellTag3_data/celltag_singlet_list.csv")

In [7]:
singlet_cells_list = pd.read_csv("/home/gzu5140/Keerthana_b1042/TwINFER/real_data/celltag_3_geo_data/celltag_singlet_list.csv")

### mapping samples/cell IDs in barcode to the samples/cells in the 10x file 

In [ ]:
# h5_files = [f for f in os.listdir(h5_dir) if f.endswith('.h5')]
# print("H5 files found:", h5_files)

# rna_samples = [s for s in barcode['sample'].unique() if 'RNA' in s]

# results = {}
# for h5_file in h5_files:
#     adata = sc.read_10x_h5(os.path.join(h5_dir, h5_file))
#     adata.var['original_gene_names'] = adata.var_names.copy()
#     adata.var_names_make_unique()
#     obs = set(adata.obs_names.str.replace(r'-\d+$', '', regex=True))
#     results[h5_file] = {}
#     for ls in rna_samples:
#         list_ids = set(barcode[barcode['sample'] == ls]['cellID'].unique())
#         results[h5_file][ls] = len(list_ids & obs)

# # Display as a matrix
# overlap_matrix = pd.DataFrame(results).T  # h5 files as rows, barcode samples as cols
# overlap_matrix = overlap_matrix[overlap_matrix.max(axis=1) > 0]  # drop zero rows
# overlap_matrix = overlap_matrix.loc[:, (overlap_matrix > 0).any()]  # drop zero cols
# print(overlap_matrix.to_string())

# Preprocessing the files

In [ ]:
h5_dir = "/home/gzu5140/Keerthana_b1042/TwINFER/real_data/celltag_3_geo_data/"

# ── 1. Load all h5 files ──────────────────────────────────────────────────────
def load_and_tag(path, key):
    adata = sc.read_10x_h5(path)
    adata.var_names_make_unique()
    adata.obs_names = [f"{key}:{bc}" for bc in adata.obs_names]
    return adata

d2_rna5 = sc.read_10x_h5(f"{h5_dir}GSM6681127_d2_5_filtered_feature_bc_matrix.h5")
d2_rna5.var_names_make_unique()
d2_rna5.obs['cellID'] = d2_rna5.obs_names.str.replace(r'-\d+$', '', regex=True)

d5_rna1 = sc.concat([
    load_and_tag(f"{h5_dir}GSM6681128_d5_1_filtered_feature_bc_matrix.h5", 's1'),
    load_and_tag(f"{h5_dir}GSM6681129_d5_2_filtered_feature_bc_matrix.h5", 's2'),
    load_and_tag(f"{h5_dir}GSM6681130_d5_3_filtered_feature_bc_matrix.h5", 's3'),
    load_and_tag(f"{h5_dir}GSM6681131_d5_4_filtered_feature_bc_matrix.h5", 's4'),
])
d5_rna1.var_names_make_unique()
d5_rna1.obs['cellID'] = d5_rna1.obs_names.str.split(':').str[-1].str.replace(r'-\d+$', '', regex=True)

d5_rna2 = sc.concat([
    load_and_tag(f"{h5_dir}GSM6681132_d5_5_filtered_feature_bc_matrix.h5", 's5'),
    load_and_tag(f"{h5_dir}GSM6681133_d5_6_filtered_feature_bc_matrix.h5", 's6'),
    load_and_tag(f"{h5_dir}GSM6681134_d5_7_filtered_feature_bc_matrix.h5", 's7'),
    load_and_tag(f"{h5_dir}GSM6681135_d5_8_filtered_feature_bc_matrix.h5", 's8'),
])
d5_rna2.var_names_make_unique()
d5_rna2.obs['cellID'] = d5_rna2.obs_names.str.split(':').str[-1].str.replace(r'-\d+$', '', regex=True)

samples_raw = {
    "d2-RNA-5": d2_rna5,
    "d5-RNA-1": d5_rna1,
    "d5-RNA-2": d5_rna2,
}


In [ ]:
# ── 2. Filter to singlets + annotate barcodes ─────────────────────────────────
def filter_and_annotate_adata(adata, metadata_df, sample_name):
    cell_ids_to_keep = (
        metadata_df[metadata_df['sample'] == sample_name]['cellID']
        .unique().tolist()
    )
    
    adata = adata.copy()
    if 'cellID' not in adata.obs.columns:
        adata.obs['cellID'] = adata.obs_names.str.replace(r'-\d+$', '', regex=True)
    mask = adata.obs['cellID'].isin(cell_ids_to_keep)
    # mask = adata.obs['cellID'].isin(cell_ids_to_keep)
    adata_filtered = adata[mask].copy()

    print(f"[{sample_name}] Singlet cell IDs to keep: {len(cell_ids_to_keep)}")
    print(f"[{sample_name}] Cells before: {adata.n_obs} → after: {adata_filtered.n_obs}")

    sample_meta = metadata_df[metadata_df['sample'] == sample_name]
    barcode_agg = (
        sample_meta.groupby('cellID')['barcode']
        .apply(list).rename('lineage_barcodes')
    )
    meta_per_cell = (
        sample_meta.groupby('cellID')
        .agg(nUMI=('nUMI', 'first'), label=('label', 'first'))
        .join(barcode_agg)
    )
    adata_filtered.obs = adata_filtered.obs.join(meta_per_cell, on='cellID', how='left')
    return adata_filtered

def is_outlier(adata, metric, nmads=5):
    M = adata.obs[metric]
    med = np.median(M)
    mad_val = median_abs_deviation(M)
    return (M < med - nmads * mad_val) | (M > med + nmads * mad_val)
from scipy.stats import median_abs_deviation
import numpy as np

def get_outlier_bounds(adata, metric, nmads=5):
    M = adata.obs[metric]
    med = np.median(M)
    mad_val = median_abs_deviation(M)
    lower = med - nmads * mad_val
    upper = med + nmads * mad_val
    return lower, upper

def qc_mad_filter(adata, sample_name, nmads=5, max_pct_mito=20):
    adata.var['mt'] = adata.var_names.str.startswith('MT-')  # use 'mt-' for mouse
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    before = adata.n_obs

    # outlier_counts = is_outlier(adata, 'total_counts', nmads)
    # outlier_genes  = is_outlier(adata, 'n_genes_by_counts', nmads)
    # outlier_mito   = adata.obs['pct_counts_mt'] > max_pct_mito
    outlier_counts = (
        is_outlier(adata, 'total_counts', nmads=2) |
        (adata.obs['total_counts'] < 4000)
    )

    outlier_genes = (
        is_outlier(adata, 'n_genes_by_counts', nmads=2) |
        (adata.obs['n_genes_by_counts'] < 200)
    )

    outlier_mito = adata.obs['pct_counts_mt'] > 10

    low_genes, high_genes = get_outlier_bounds(adata, 'n_genes_by_counts', nmads)
    low_genes = max(low_genes, 200)
    low_counts, high_counts = get_outlier_bounds(adata, 'total_counts', nmads)
    low_counts = max(low_counts, 4000)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # ── n_genes_by_counts ─────────────────────────────
    axes[0].scatter(
        adata.obs["n_genes_by_counts"],
        adata.obs["total_counts"],
        c=adata.obs["pct_counts_mt"],
        s=5,
        alpha=0.6,
        cmap="viridis"
    )
    axes[0].axvline(low_genes, color="red", linestyle="--", label="lower bound")
    axes[0].axvline(high_genes, color="red", linestyle="--", label="upper bound")
    axes[0].axhline(low_counts, color="red", linestyle="--", label="lower bound")
    axes[0].axhline(high_counts, color="red", linestyle="--", label="upper bound")
    axes[0].set_title('n_genes_by_counts')
    axes[0].set_xlabel('n_genes_by_counts')
    axes[0].set_ylabel('count')
    axes[0].legend()

    # ── total_counts ─────────────────────────────
    axes[1].hist(adata.obs['total_counts'], bins=50, color="steelblue")
    axes[1].axvline(low_counts, color="red", linestyle="--", label="lower bound")
    axes[1].axvline(high_counts, color="red", linestyle="--", label="upper bound")
    axes[1].set_title('total_counts')
    axes[1].set_xlabel('total_counts')
    axes[1].set_ylabel('count')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f"[{sample_name}] Outlier breakdown:")
    print(f"  total_counts outliers:      {outlier_counts.sum()}")
    print(f"  n_genes_by_counts outliers: {outlier_genes.sum()}")
    print(f"  high mito%:                 {outlier_mito.sum()}")
    
    fail_any = outlier_counts | outlier_genes | outlier_mito
    adata.obs['qc_pass'] = ~fail_any

    adata_filt = adata[adata.obs['qc_pass']].copy()

    print(f"[{sample_name}] QC: {before} → {adata_filt.n_obs} cells ({before - adata_filt.n_obs} removed)")
    return adata_filt

# Usage
# ── 3. QC filtering ───────────────────────────────────────────────────────────
def qc_filter(adata, sample_name,
              min_genes=200, max_genes=6000,
              min_counts=500, max_counts=30000,
              max_pct_mito=20):
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    before = adata.n_obs

    # Build individual boolean masks for each criterion
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(adata.obs['n_genes_by_counts'])
    axes[0].axvline(min_genes, linestyle = "--")
    axes[0].axvline(max_genes, linestyle = "--")
    axes[0].set_title('n_genes_by_counts')
    axes[0].set_xlabel('n_genes_by_counts')
    axes[0].set_ylabel('count')

    axes[1].hist(adata.obs['total_counts'])
    axes[1].axvline(min_counts, linestyle = "--")
    axes[1].axvline(max_counts, linestyle = "--")
    axes[1].set_title('total_counts')
    axes[1].set_xlabel('total_counts')
    axes[1].set_ylabel('count')

    plt.tight_layout()
    plt.show()
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))

    sc = ax

    ax.axvline(min_genes, linestyle="--", color="red")
    ax.axvline(max_genes, linestyle="--", color="red")
    ax.axhline(min_counts, linestyle="--", color="blue")
    ax.axhline(max_counts, linestyle="--", color="blue")

    plt.colorbar(sc, ax=ax, label="% mitochondrial")
    ax.set_xlabel("n_genes_by_counts")
    ax.set_ylabel("total_counts")
    ax.set_title("QC scatter colored by mito%")

    plt.tight_layout()
    plt.show()

    fail_min_genes  = adata.obs['n_genes_by_counts'] < min_genes
    fail_max_genes  = adata.obs['n_genes_by_counts'] > max_genes
    fail_min_counts = adata.obs['total_counts'] < min_counts
    fail_max_counts = adata.obs['total_counts'] > max_counts
    fail_mito       = adata.obs['pct_counts_mt'] > max_pct_mito
    
    # Report how many cells fail each criterion (not mutually exclusive —
    # a cell can fail more than one)
    print(f"[{sample_name}] QC breakdown (before filtering, {before} cells):")
    print(f"  Too few genes   (<{min_genes}):        {fail_min_genes.sum()}")
    print(f"  Too many genes  (>{max_genes}):        {fail_max_genes.sum()}")
    print(f"  Too few counts  (<{min_counts}):        {fail_min_counts.sum()}")
    print(f"  Too many counts (>{max_counts}):        {fail_max_counts.sum()}")
    print(f"  High mito%      (>{max_pct_mito}%):     {fail_mito.sum()}")

    fail_any = fail_min_genes | fail_max_genes | fail_min_counts | fail_max_counts | fail_mito
    print(f"  Total unique cells failing ≥1 criterion: {fail_any.sum()}")

    keep = ~fail_any
    adata = adata[keep].copy()
    print(f"[{sample_name}] QC: {before} → {adata.n_obs} cells ({before - adata.n_obs} removed)")
    return adata


filtered = {
    name: filter_and_annotate_adata(adata, singlet_cells_list, name)
    for name, adata in samples_raw.items()
}

qc_passed = {
    name: qc_mad_filter(adata, name,
                    # min_genes=200, max_genes=6000,
                    # min_counts=500, max_counts=30000,
                    max_pct_mito=10)
    for name, adata in filtered.items()
}

# ── 5. Concatenate ────────────────────────────────────────────────────────────
for name in qc_passed:
    qc_passed[name].var_names_make_unique()



In [ ]:
combined = sc.concat(qc_passed, label='sample', merge='same')
print(f"Combined: {combined.n_obs} cells × {combined.n_vars} genes")
combined.obs['match_key'] = combined.obs['sample'].astype(str) + '_' + combined.obs_names.astype(str)
combined.obs['match_key'] = combined.obs['match_key'].str[:-2]

In [ ]:
# ── 1. Parse CSV, keeping barcode+suffix together ────────────────────────────
clone_df = pd.read_csv('/home/gzu5140/Keerthana_b1042/TwINFER/real_data/celltag_3_geo_data/hsc.rna&atac.r1&2_master_v2.csv')

split_bc = clone_df['cell.bc'].str.rsplit('-', n=2, expand=True)
clone_df['sample_assay'] = split_bc[0]
clone_df['barcode']      = split_bc[1]           # raw 16bp barcode, no suffix
clone_df['suffix']       = split_bc[2]           # "1","2","3","4"

sa_split = clone_df['sample_assay'].str.rsplit('-', n=1, expand=True)
clone_df['sample'] = sa_split[0]   # "d2_5","d5r1","d5r2"
clone_df['assay']  = sa_split[1]   # "rna"

clone_rna = clone_df[clone_df['assay'] == 'rna'].copy()

# ── 2. Reconstruct the s1..s8 lane label to match your load_and_tag scheme ──
lane_map = {
    ('d2_5', '1'): None,          # d2_5 loaded as single h5, no lane prefix used
    ('d5r1', '1'): 's1', ('d5r1', '2'): 's2', ('d5r1', '3'): 's3', ('d5r1', '4'): 's4',
    ('d5r2', '1'): 's5', ('d5r2', '2'): 's6', ('d5r2', '3'): 's7', ('d5r2', '4'): 's8',
}
clone_rna['lane'] = clone_rna.apply(lambda r: lane_map.get((r['sample'], r['suffix'])), axis=1)

# ── 3. Build match_key matching each combined sub-object's obs_names format ─
# d2_5: obs_names = raw barcode with original -N suffix (untouched)
# d5_rna1 / d5_rna2: obs_names = "sX:barcode-N" (prefix + original suffix kept in obs_names,
#                     but cellID column has suffix stripped — use obs_names, not cellID!)

sample_map = {'d2_5': 'd2-RNA-5', 'd5r1': 'd5-RNA-1', 'd5r2': 'd5-RNA-2'}
clone_rna['sample_combined'] = clone_rna['sample'].map(sample_map)

def build_match_key(row):
    if row['sample'] == 'd2_5':
        return f"{row['sample_combined']}_{row['barcode']}"
    else:
        return f"{row['sample_combined']}_{row['lane']}:{row['barcode']}"

clone_rna['match_key'] = clone_rna.apply(build_match_key, axis=1)

print("Duplicates:", clone_rna['match_key'].duplicated().sum())  # should be 0 now

# ── 4. Build the same key structure from combined.obs_names (NOT cellID) ────
key_to_clone = clone_rna.set_index('match_key')['clone.id']
combined.obs['clone_id'] = combined.obs['match_key'].map(key_to_clone)

n_matched = combined.obs['clone_id'].notna().sum()
print(f"Matched {n_matched}/{combined.n_obs} cells to a clone ID")
print(combined.obs.groupby('sample')['clone_id'].apply(lambda x: x.notna().sum()))

In [13]:
# HVGs on raw counts
sc.pp.highly_variable_genes(
    combined,
    n_top_genes=3000,
    batch_key="sample",
    flavor="seurat_v3"
)

# Preserve raw counts
combined.layers["counts"] = combined.X.copy()

# Normalize and log
combined.raw = combined
# sc.pp.normalize_total(combined, target_sum=1e4)
# sc.pp.log1p(combined)

# # # Store log-normalized data

# # # Scale
# sc.pp.scale(combined, max_value=10)

# # PCA
# sc.tl.pca(combined, n_comps=30, svd_solver="arpack")
scclr.pp.pflog(combined, target="auto")      # -> adata.layers["pflog"] + obs center
scclr.tl.pca(combined, n_comps=50)             # -> adata.obsm["X_pca"], varm["PCs"], uns["pca"]
# Neighbors / clustering / UMAP
sc.pp.neighbors(combined, n_neighbors=15)
sc.tl.louvain(combined, resolution=0.5)
sc.tl.umap(combined)

In [ ]:
sc.pl.umap(combined, color='sample')

In [ ]:
sc.pl.umap(combined, color=['louvain', 'sample'])

# or a contingency table
import pandas as pd

pd.crosstab(
    combined.obs['louvain'],
    combined.obs['sample'],
    normalize='columns'
)

# Twinfer analysis similar to LARRY

In [16]:
gene_list_Regulator_TF = ['Gata1', 'Gata2', 'Gfi1', 'Fli1', 'Spi1', 'Tal1',  'Cebpa', 'Jun', 'Egr1', 'Nab2', 'Klf1', 'Zfpm1'] #TF involved in hematopoiesis regulation
curr_gene_list = gene_list_Regulator_TF
gene_set_name = "Regulator_TF"

In [ ]:
import numpy as np

gene_list_Regulator_TF = ['Gata1', 'Gata2', 'Gfi1', 'Fli1', 'Spi1', 'Tal1',  'Cebpa', 'Jun', 'Egr1', 'Nab2', 'Klf1', 'Zfpm1'] #TF involved in hematopoiesis regulation
curr_gene_list = gene_list_Regulator_TF
gene_set_name = "Regulator_TF"

combined.obs["timepoint"] = np.where(
    combined.obs["sample"].str.contains("d2"),
    "d2",
    "d5"
)
adata = combined.copy()
del combined
import gc
gc.collect()

In [ ]:
import matplotlib.pyplot as plt
import math
import pandas as pd


genes = curr_gene_list
n_cols = 3
n_rows = math.ceil(len(genes) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, gene in enumerate(genes):
    # Extract raw counts for the gene
    gene_idx = adata.raw.var_names.get_loc(gene)
    data = pd.Series(adata.raw.X[:, gene_idx].toarray().flatten()).astype(int)

    data = data[data > 0]
    
    n_unique = data.nunique()
    
    axes[i].hist(data)
    axes[i].set_xlabel(gene)
    
    if n_unique <= 3:
        pcts = data.value_counts(normalize=True) * 100
        pct_str = ", ".join([f"{v}: {p:.1f}%" for v, p in pcts.items()])
        axes[i].set_title(f'{gene} | {pct_str}')
    else:
        axes[i].set_title(f'{gene} | n_unique={n_unique}')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
#Check that all of these genes are present in the dataset (since sometimes they have alternate names. E.g. Pu1 is names Spi1)
gene_list_check = gene_list_Regulator_TF

all_genes = {g.lower() for g in adata.var_names}
gene_list_check =  [g.lower() for g in gene_list_check]
found = [a for a in gene_list_check if a in all_genes]
not_found = [a for a in gene_list_check if a not in all_genes]
print(f"Found {len(found)} genes out of {len(gene_list_check)}")
if found:
    print("✅ Found gene(s):", found)

print("❌ Not found in dataset", not_found)

In [21]:
# ============================
#   1. Distance computation
# =============
# ===============
def compute_distances(adata):
    """
    Compute pairwise transcriptomic distances assuming cells are sorted
    in twin order: (cell0_A, cell0_B, cell1_A, cell1_B, ...)
    and that each consecutive pair belongs to the same clone.
    """
    pca = adata.obsm['X_pca']
    clone_ids = adata.obs['clone_id'].values

    # sanity check: even number of cells
    if pca.shape[0] % 2 != 0:
        raise ValueError("Number of cells must be even for twin pairing.")

    # check clone_id matching
    clone_left  = clone_ids[::2]
    clone_right = clone_ids[1::2]

    if not np.all(clone_left == clone_right):
        bad = np.where(clone_left != clone_right)[0]
        raise ValueError(
            f"Found {len(bad)} twin pairs with mismatched clone_id. "
            f"First few bad indices: {bad[:5]}"
        )

    # compute distances
    deltas = pca[::2] - pca[1::2]
    return np.linalg.norm(deltas, axis=1)


# ============================
#   2. Significance symbols
# ============================
def get_significance_symbol(pval):
    if pval > 0.05:
        return 'ns'
    elif pval <= 0.0001:
        return '****'
    elif pval <= 0.001:
        return '***'
    elif pval <= 0.01:
        return '**'
    else:
        return '*'

# ============================
#   3. Stat annotation for boxplots
# ============================
def add_stat_annotation(ax, category_a, category_b, label, subset,
                        y_offset=0.05, level=0):
    """
    Draws a significance bracket ABOVE vertical boxplots with clip turned OFF.
    """

    # X-axis category names
    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]

    if category_a not in xticklabels or category_b not in xticklabels:
        return

    x1 = xticklabels.index(category_a)
    x2 = xticklabels.index(category_b)
    x_min, x_max = min(x1, x2), max(x1, x2)

    # Height for bracket
    ymax = subset['Transcriptomic distance [a.u.]'].max()
    h = ymax * (1 + y_offset + level * 0.12)

    # Bracket line (no clipping)
    ax.plot(
        [x_min, x_min, x_max, x_max],
        [h, h * 1.05, h * 1.05, h],
        lw=1.5,
        color='black',
        clip_on=False
    )

    # Label (no clipping)
    ax.text(
        (x_min + x_max) / 2,
        h * 1.12,
        label,
        ha='center',
        va='bottom',
        fontsize=12,
        clip_on=False
    )

def plot_vertical_boxplots(df_distances, save_dir = None):

    # plt.subplots_adjust(top=0.83)  # Ensure title + brackets never clip

    timepoints = df_distances['timepoint'].unique()
    n_col_plot = min(len(timepoints), 3)
    n_row_plot = max(1, n_col_plot//3)
    fig, axes = plt.subplots(n_row_plot, n_col_plot, figsize=(15, 6), sharey=True)

    for ax, timepoint in zip(axes, timepoints):

        subset = df_distances[df_distances['timepoint'] == timepoint]

        # -----------------------------
        # Vertical boxplot
        # -----------------------------
        bp = sns.boxplot(
            data=subset,
            x='pair_type',
            y='Transcriptomic distance [a.u.]',
            hue='pair_type',
            palette={'clonal pairs': 'lightgray', 'random pairs': 'gray'},
            ax=ax
        )

        # Disable clipping for every boxplot artist
        for artist in ax.artists + ax.lines:
            artist.set_clip_on(False)

        sns.despine(right=True, top=True, ax=ax)

        ax.set_xlabel('')
        ax.set_ylabel('transcriptomic distance [a.u.]')
        ax.set_title(timepoint, pad=30)  # higher title
        # -----------------------------
        # Mann-Whitney U-test
        # -----------------------------
        clonal = subset[subset['pair_type'] == 'clonal pairs']['Transcriptomic distance [a.u.]']
        random = subset[subset['pair_type'] == 'random pairs']['Transcriptomic distance [a.u.]']
        print(len(clonal), len(random))
        _, pval = mannwhitneyu(clonal, random, alternative='two-sided')
        label = get_significance_symbol(pval)
        # # -----------------------------
        # # Add significance bracket
        # # -----------------------------
        add_stat_annotation(
            ax=ax,
            category_a='clonal pairs',
            category_b='random pairs',
            label=label,
            subset=subset,
            y_offset=0.05,
            level=0
        )


    # Legend outside (no clipping)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()
    if save_dir:
        plt.savefig(os.path.join(save_dir, 'twin_random_PCA_dist_day_2.svg'),
                    format='svg',
                    facecolor='none', edgecolor='none', transparent=True)
        plt.savefig(os.path.join(save_dir, 'twin_random_PCA_dist_day_2.pdf'),
                    format='pdf',
                    facecolor='none', edgecolor='none', transparent=True)

In [22]:
t1 = "d2"
t2 = "d5"

#Subset the data into different time points - includes both barcoded and not barcoded cells (hence, all)
adata.obs['cell_id'] = adata.obs.index
adata_t1_all = adata[(adata.obs['timepoint'] == t1)].copy()
adata_t2_all = adata[(adata.obs['timepoint'] == t2)].copy()

# All cells at time t1 to calculate gene correlation
def make_all_cells_table(adata_t, timepoint, gene_subset, curr_gene_list):
    df = pd.DataFrame({
        'clone_id': adata_t.obs['lineage_barcodes'].apply(
            lambda x: '|'.join(x) if isinstance(x, list) else str(x)
        ).values,
        'cell_id': adata_t.obs.index.values,  # use actual cell index, not barcodes
    })

    df['pair_id'] = df['cell_id'].astype(str) + f"_single_{timepoint}"
    df['replicate'] = 1
    df['time_step'] = timepoint

    df[gene_subset] = adata_t[df.cell_id, curr_gene_list].X.toarray()

    return df

gene_subset = [s + '_mRNA' for s in curr_gene_list]
t1_data_all_cells = make_all_cells_table(
    adata_t1_all, t1, gene_subset, curr_gene_list
)
t2_data_all_cells = make_all_cells_table(
    adata_t2_all, t2, gene_subset, curr_gene_list
)


In [ ]:
clone_sizes

In [ ]:
clone_sizes = adata_t1.obs[adata_t1.obs['clone_id']!="nan"]['clone_id'].value_counts()

plt.figure(figsize=(8, 5))
plt.hist(clone_sizes, bins=range(1, clone_sizes.max() + 2), align='left', edgecolor='black')
plt.xlabel('Clone size (# cells)')
plt.ylabel('Number of clones')
plt.title('Day 2 clone sizes')
plt.xticks(range(1, clone_sizes.max() + 1))
plt.show()

In [ ]:
clone_sizes = adata_t2.obs[adata_t2.obs['clone_id']!="nan"]['clone_id'].value_counts()

plt.figure(figsize=(8, 5))
plt.hist(clone_sizes, bins=range(1, clone_sizes.max() + 2), align='left', edgecolor='black')
plt.xlabel('Clone size (# cells)')
plt.ylabel('Number of clones')
plt.title('Day 5 clone sizes')
plt.xticks(range(1, clone_sizes.max() + 1, 2))
plt.show()

In [ ]:
#Identifying barcoded cells at time t1
adata_t1 = t1_data_all_cells[(t1_data_all_cells['clone_id'] != -1)].copy()
# adata_t1_clones_undiff = adata_t1[adata_t1.obs['Cell type annotation'] == 'Undifferentiated']

# Print the results
print(f"Number of barcoded cells day 2: {adata_t1.shape[0]}")

adata_t2 = t2_data_all_cells[(t2_data_all_cells['clone_id'] != -1)].copy()

print(f"Number of barcoded cells day 5: {adata_t2.shape[0]}")

In [ ]:
import itertools
#Twin pairs at each time point and across time point
adata_t1 = adata_t1_all[(adata_t1_all.obs['clone_id'] != -1)].copy()
adata_t2 = adata_t2_all[(adata_t2_all.obs['clone_id'] != -1)].copy()


# Save cell IDs in .obs
adata_t1.obs['cell_id'] = adata_t1.obs_names
adata_t2.obs['cell_id'] = adata_t2.obs_names

# Pick subset of genes
gene_subset = [s + '_mRNA' for s in curr_gene_list]

# Create tables for t1, t2 and t3 twin pairs
for adata_t, timepoint in zip([adata_t1,adata_t2], ['t1','t2']):
    rows = []
    for clone_id, group in adata_t.obs.groupby('clone_id'):
        cells = group['cell_id'].tolist()
        pair_counter = 0
        for c1, c2 in itertools.combinations(cells, 2):
            pair_id = f"{clone_id}_p{pair_counter}_{timepoint}"
            rows.append({
                'clone_id': clone_id,
                'pair_id': pair_id,
                'cell_id': c1,
                'replicate': 1
            })
            rows.append({
                'clone_id': clone_id,
                'pair_id': pair_id,
                'cell_id': c2,
                'replicate': 2
            })
            pair_counter += 1

    if timepoint == 't1':
        t1_data = pd.DataFrame(rows)
    elif timepoint == 't2':
        t2_data = pd.DataFrame(rows)

t1_data['time_step'] = np.repeat(t1, len(t1_data))
t2_data['time_step'] = np.repeat(t2, len(t2_data))

t1_data[gene_subset] = adata_t1[t1_data.cell_id, curr_gene_list].X.toarray()
t2_data[gene_subset] = adata_t2[t2_data.cell_id, curr_gene_list].X.toarray()

# ### Create tables for across t twin pairs
across_t_clones = list(set(adata_t1.obs.clone_id).intersection(adata_t2.obs.clone_id))
adata_t1_sub = adata_t1[adata_t1.obs.clone_id.isin(across_t_clones)]
adata_t2_sub = adata_t2[adata_t2.obs.clone_id.isin(across_t_clones)]

rows_t1 = []
rows_t2 = []
for clone_id in across_t_clones:
    cells_t1 = adata_t1_sub[adata_t1_sub.obs.clone_id == clone_id].obs['cell_id'].tolist()
    cells_t2 = adata_t2_sub[adata_t2_sub.obs.clone_id == clone_id].obs['cell_id'].tolist()
    pair_counter = 0
    for cell_t1 in cells_t1:
        for cell_t2 in cells_t2:
            pair_id = f"{clone_id}_p{pair_counter}_across_t"
            rows_t1.append({
                'clone_id': clone_id,
                'pair_id': pair_id,
                'cell_id': cell_t1,
                'replicate': 1,
                'time_step': t1
            })
            rows_t2.append({
                'clone_id': clone_id,
                'pair_id': pair_id,
                'cell_id': cell_t2,
                'replicate': 2,
                'time_step': t2
            })

            pair_counter += 1

across_t_data_t1 = pd.DataFrame(rows_t1)
across_t_data_t2 = pd.DataFrame(rows_t2)

across_t_data_t1[gene_subset] = adata_t1[across_t_data_t1.cell_id, curr_gene_list].X.toarray()
across_t_data_t2[gene_subset] = adata_t2[across_t_data_t2.cell_id, curr_gene_list].X.toarray()
across_t_data = pd.concat([across_t_data_t1, across_t_data_t2])

print(f"Number of t1 twins: {int(t1_data.shape[0]/2)}")
print(f"Number of t2 twins: {int(t2_data.shape[0]/2)}")
print(f"Number of across t twins: {int(across_t_data_t1.shape[0])}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
adata_t1.obs['clone_id'] = adata_t1.obs['clone_id'].astype(str)
adata_t2.obs['clone_id'] = adata_t2.obs['clone_id'].astype(str)

t1_clones = set(adata_t1.obs['clone_id'][adata_t1.obs['clone_id'] != "nan"].dropna())
t2_clones = set(adata_t2.obs['clone_id'][adata_t2.obs['clone_id'] != "nan"].dropna())

across_t_clones = list(t1_clones.intersection(t2_clones))
print(f"Shared clones: {len(across_t_clones)}")

size_df = pd.DataFrame({
    'd2_size': adata_t1_sub.obs['clone_id'].astype(str).value_counts(),
    'd5_size': adata_t2_sub.obs['clone_id'].astype(str).value_counts()
}).fillna(0).astype(int)

size_df = size_df.reindex(across_t_clones)
print(size_df.isna().sum())  # should now be 0, 0

# # ── Per-clone size table ──────────────────────────────────────────────────
# size_df = pd.DataFrame({
#     'd2_size': adata_t1_sub.obs['clone_id'].value_counts(),
#     'd5_size': adata_t2_sub.obs['clone_id'].value_counts()
# }).fillna(0).astype(int)
# size_df = size_df.reindex(across_t_clones)

# ── Number of possible across-time cell pairs per clone = n * m ────────────
size_df['n_pairs'] = size_df['d2_size'] * size_df['d5_size']

# ── Scatter plot colored by n_pairs ──────────────────────────────────────────
plt.figure(figsize=(7, 6))
sc_plot = plt.scatter(
    size_df['d2_size'], size_df['d5_size'],
    c=size_df['n_pairs'], cmap='viridis',
    s=60, edgecolor='black', alpha=0.85
)
plt.colorbar(sc_plot, label='# possible across-time pair')
plt.xlabel('Day 2 clone size (# cells)')
plt.ylabel('Day 5 clone size (# cells)')
plt.title('Clone size: Day 2 vs Day 5, colored by pair count')

max_val = max(size_df['d2_size'].max(), size_df['d5_size'].max())
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compute distances in PC space
distances_t1 = compute_distances(adata_t1[t1_data.cell_id])
distances_t2 = compute_distances(adata_t2[t2_data.cell_id])

# Random
np.random.seed(42)

def random_distances(adata, n_pairs):
    idx = np.arange(adata.n_obs)
    pairs = np.random.choice(idx, size=(n_pairs, 2), replace=True)
    pairs = pairs[pairs[:, 0] != pairs[:, 1]]
    while pairs.shape[0] < n_pairs:
        extra = np.random.choice(idx, size=(n_pairs - pairs.shape[0], 2), replace=True)
        extra = extra[extra[:, 0] != extra[:, 1]]
        pairs = np.vstack([pairs, extra])
    return np.linalg.norm(adata.obsm["X_pca"][pairs[:, 0]] - adata.obsm["X_pca"][pairs[:, 1]], axis=1)

distances_random_t1 = random_distances(adata_t1, len(distances_t1))
distances_random_t2 = random_distances(adata_t2, len(distances_t2))

# Create dataframe for plotting
df_distances = pd.DataFrame({
    'Transcriptomic distance [a.u.]': np.concatenate([distances_t1, distances_t2,
                                distances_random_t1, distances_random_t2]),
    'pair_type': (['clonal pairs'] * len(distances_t1) +
                  ['clonal pairs'] * len(distances_t2) +
                  ['random pairs'] * len(distances_random_t1) +
                  ['random pairs'] * len(distances_random_t2)) ,
    
    'timepoint': (['Day 2'] * len(distances_t1) +
                  ['Day 5'] * len(distances_t2) +
                  ['Day 2'] * len(distances_random_t1) +
                  ['Day 5'] * len(distances_random_t2) 
    )
})

In [ ]:
plot_vertical_boxplots(df_distances, save_dir = None)

## Run Twinfer

In [29]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numba
import tqdm
import scipy
import os
import sys
import joblib
import scanpy as sc
import os
import itertools
from itertools import product
from collections import defaultdict
from itertools import combinations
from scipy.stats import mannwhitneyu
from pathlib import Path
# from adjustText import adjust_text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
import json

In [42]:
# Calculation functions
import importlib
import sys
import os

sys.path.append("/home/gzu5140/Keerthana_b1042/TwINFER/code/TwINFER")
from TwINFER_function_scripts import correlation_analysis_functions
from TwINFER_function_scripts import correlation_analysis_helpers

importlib.reload(correlation_analysis_functions)
importlib.reload(correlation_analysis_helpers)

from TwINFER_function_scripts.correlation_analysis_functions import (

    calculate_pairwise_gene_gene_correlation_matrix,
    check_system_in_steady_state,
    check_gene_gene_correlation_threshold,
    calculate_twin_random_pair_correlations,
    differentiate_single_state_reg_and_multiple_states,
    identify_reg_if_multiple_states,
    get_cross_correlations,
    identify_actual_directed_edges
)

# Helper functions
from TwINFER_function_scripts.correlation_analysis_helpers import (
    extract_param_index,
    read_input_matrix,
    get_param_data,
    plot_matrix_as_heatmap,
    print_summary,
    plot_network
)

### Plot formatting

In [41]:
#colormap helper function

def make_reds_blues_colormap(vmin=-0.05, vmax=0.18):
    """Custom red–white–blue colormap with pure white at 0, asymmetric."""
    # Calculate where 0 falls in the range [vmin, vmax]
    zero_position = max(0, (0 - vmin) / (vmax - vmin))
    
    # Number of colors for each segment (proportional to range)
    n_total = 256
    n_reds = int(zero_position * n_total)  # colors from vmin to 0
    n_blues = n_total - n_reds  # colors from 0 to vmax
    
    # Calculate intensity based on actual distance from zero
    # For reds: map from vmin to 0, so max intensity at vmin
    red_intensity = abs(vmin) / max(abs(vmin), abs(vmax))  # 0.05/0.18 ≈ 0.28
    # For blues: map from 0 to vmax, so max intensity at vmax  
    blue_intensity = abs(vmax) / max(abs(vmin), abs(vmax))  # 0.18/0.18 = 1.0
    
    # Create color arrays with scaled intensities
    reds = plt.cm.Reds(np.linspace(0.8 * red_intensity, 0, n_reds))  # scaled dark to light red
    whites = np.ones((1, 4))  # pure white at 0
    blues = plt.cm.Blues(np.linspace(0, 0.8 * blue_intensity, n_blues))  # light to scaled dark blue
    
    colors = np.vstack((reds, whites, blues))
    return LinearSegmentedColormap.from_list('RedsBlues', colors)

### Data cleanup before analysis

In [43]:
# Drop column clone_id and rename pair_id to clone_id
t1_data.drop(columns=['clone_id'], inplace=True)
t1_data.rename(columns={'pair_id': 'clone_id'}, inplace=True)
t2_data.drop(columns=['clone_id'], inplace=True)
t2_data.rename(columns={'pair_id': 'clone_id'}, inplace=True)

across_t_data.drop(columns=['clone_id'], inplace=True)
across_t_data.rename(columns={'pair_id': 'clone_id'}, inplace=True)

t1_clones = t1_data.clone_id.values
t2_clones = t2_data.clone_id.values
across_t_clones = across_t_data.clone_id.values

# Subset directly
t1_twins = t1_data
t2_twins = t2_data

# Across_t: pick the pair_id from the two points - so all pairs of twins are counted
# One cell per clone at t1
across_t_twin1 = across_t_data[across_t_data.time_step == t1]
across_t_twin2 = across_t_data[across_t_data.time_step == t2]

# Reset index for cleanliness
t1_twins = t1_twins.reset_index(drop=True)
t2_twins = t2_twins.reset_index(drop=True)

across_t_twin1 = across_t_twin1.reset_index(drop=True)
across_t_twin2 = across_t_twin2.reset_index(drop=True)

all_t1_measurements = (
    pd.concat([t1_twins, across_t_twin1], ignore_index=True)
      .drop_duplicates(subset="cell_id", keep="first")
)

all_t2_measurements = (
    pd.concat([t2_twins, across_t_twin2], ignore_index=True)
      .drop_duplicates(subset="cell_id", keep="first")
)



In [ ]:
import matplotlib.pyplot as plt
import math

genes = gene_subset  # your gene list

n_cols = 2
n_rows = math.ceil(len(genes) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, gene in enumerate(genes):
    data = t1_twins[gene]
    n_unique = data.nunique()
    
    axes[i].hist(data)
    axes[i].set_xlabel(gene)
    
    if n_unique <= 2:
        pcts = data.value_counts(normalize=True) * 100
        pct_str = ", ".join([f"{v}: {p:.1f}%" for v, p in pcts.items()])
        axes[i].set_title(f'{gene} | n_unique={n_unique}\n{pct_str}')
    else:
        axes[i].set_title(f'{gene} | n_unique={n_unique}')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Sets of cell IDs
t1_twin_cells = set(t1_twins["cell_id"])
t1_across_cells = set(across_t_twin1["cell_id"])

t2_twin_cells = set(t2_twins["cell_id"])
t2_across_cells = set(across_t_twin2["cell_id"])

# Overlaps
overlap_t1 = t1_twin_cells & t1_across_cells
overlap_t2 = t2_twin_cells & t2_across_cells

print("T1 overlap (t1 ∩ across):", len(overlap_t1))
print("T2 overlap (twin ∩ across):", len(overlap_t2))


## Step 1: Gene-gene correlation

In [46]:
# Define input parameters
plot_correlation_matrices_as_heatmap = True
have_any_output = False
p_val_threshold_scrambled_gene_correlation = 0.02
show_scrambled_distribution_gene_correlation = True
z_score_threshold_two_states = 10
n_shuffles=10000

In [ ]:
# # --- Step 1: Pairwise gene-gene correlations at t1: day 2 ---
import matplotlib.pyplot as plt
import math
import pandas as pd

import numpy as np

gene_list_Regulator_TF = ['Gata1', 'Gata2', 'Gfi1', 'Fli1', 'Spi1', 'Tal1',  'Cebpa', 'Jun', 'Egr1', 'Nab2', 'Klf1', 'Zfpm1'] #TF involved in hematopoiesis regulation
curr_gene_list = gene_list_Regulator_TF
gene_set_name = "Regulator_TF"

combined.obs["timepoint"] = np.where(
    combined.obs["sample"].str.contains("d2"),
    "d2",
    "d5"
)
adata = combined.copy()
del combined
import gc
gc.collect()

genes = curr_gene_list
n_cols = 3
n_rows = math.ceil(len(genes) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, gene in enumerate(genes):
    # Extract raw counts for the gene
    gene_idx = adata.raw.var_names.get_loc(gene)
    data = pd.Series(adata.raw.X[:, gene_idx].toarray().flatten()).astype(int)

    data = data[data > 0]
    
    n_unique = data.nunique()
    
    axes[i].hist(data)
    axes[i].set_xlabel(gene)
    
    if n_unique <= 3:
        pcts = data.value_counts(normalize=True) * 100
        pct_str = ", ".join([f"{v}: {p:.1f}%" for v, p in pcts.items()])
        axes[i].set_title(f'{gene} | {pct_str}')
    else:
        axes[i].set_title(f'{gene} | n_unique={n_unique}')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()
pairwise_gene_gene_correlation_matrix_t1 = calculate_pairwise_gene_gene_correlation_matrix(
    all_t1_measurements
    , curr_gene_list
)
print(pairwise_gene_gene_correlation_matrix_t1)
no_regulation_t1, potential_regulation_t1, threshold, _ = check_gene_gene_correlation_threshold(
    all_t1_measurements, pairwise_gene_gene_correlation_matrix_t1, curr_gene_list, n_shuffles = n_shuffles, use_scramble = True, p_val_threshold = p_val_threshold_scrambled_gene_correlation, verbose = show_scrambled_distribution_gene_correlation, n_cores_to_use=18
)

# print(no_regulation)
if plot_correlation_matrices_as_heatmap:
    plot_matrix_as_heatmap(corr_matrix=pairwise_gene_gene_correlation_matrix_t1, gene_list=curr_gene_list, no_regulation=no_regulation_t1, potential_regulation=potential_regulation_t1,
        title=f"Gene-gene correlations for {gene_set_name} - day 2", add_gene_labels=True, add_time=False, gray_out_no_reg=False, black_out_self=True
    )

In [54]:
pairwise_gene_gene_correlation_matrix_t1.to_csv(f"{output_path}/pairwise_gene_gene_correlation_matrix_t1_r2.csv")

In [ ]:
# --- Step 1: Pairwise gene-gene correlations at t2: day 5 ---
pairwise_gene_gene_correlation_matrix_t2 = calculate_pairwise_gene_gene_correlation_matrix(
    all_t2_measurements, curr_gene_list
)
no_regulation_t2, potential_regulation_t2, _, _= check_gene_gene_correlation_threshold(
    all_t2_measurements, pairwise_gene_gene_correlation_matrix_t2, curr_gene_list, use_scramble = True, p_val_threshold = p_val_threshold_scrambled_gene_correlation, verbose = show_scrambled_distribution_gene_correlation,  n_cores_to_use=19
)

# print(no_regulation)
if plot_correlation_matrices_as_heatmap:
    plot_matrix_as_heatmap(corr_matrix=pairwise_gene_gene_correlation_matrix_t2, gene_list=curr_gene_list, no_regulation=no_regulation_t2, potential_regulation=potential_regulation_t2,
        title=f"Gene-gene correlations for {gene_set_name} - day 5", add_gene_labels=True, add_time=False, gray_out_no_reg=False, black_out_self = True
    )

In [56]:
pairwise_gene_gene_correlation_matrix_t2.to_csv(f"{output_path}/pairwise_gene_gene_correlation_matrix_t2_r2.csv")

In [57]:
# === Combine and save all timepoint results ===
rows = []

for tp, (no_reg, pot_reg, corr_mat) in {
    "t1": (no_regulation_t1, potential_regulation_t1, pairwise_gene_gene_correlation_matrix_t1),
    "t2": (no_regulation_t2, potential_regulation_t2, pairwise_gene_gene_correlation_matrix_t2),
}.items():

    all_pairs = set(tuple(sorted(p)) for p in no_reg + pot_reg)

    for g1, g2 in all_pairs:
        # lookup correlation (try both orders)
        if g1 in corr_mat.index and g2 in corr_mat.columns:
            corr_val = corr_mat.loc[g1, g2]
        elif g2 in corr_mat.index and g1 in corr_mat.columns:
            corr_val = corr_mat.loc[g2, g1]
        else:
            corr_val = None

        pair_sorted = tuple(sorted((g1, g2)))
        if pair_sorted in [tuple(sorted(p)) for p in pot_reg]:
            category = "potential_regulation"
        elif pair_sorted in [tuple(sorted(p)) for p in no_reg]:
            category = "no_regulation"
        else:
            category = "uncategorized"

        rows.append([g1, g2, corr_val, category, tp])

# Create DataFrame
df = pd.DataFrame(rows, columns=["gene_1", "gene_2", "correlation", "category", "timepoint"])

# Define output filename with timestamp
outfile = f"{output_path}/gene_pair_results_{gene_set_name}_r2.csv"

# Save file
df.to_csv(outfile, index=False)

# Print confirmation with readable date/time


In [ ]:
# === Load saved CSV ===
df = pd.read_csv(outfile)

# === Reconstruct lists and matrices ===
no_regulation = {}
potential_regulation = {}
pairwise_gene_gene_correlation_matrix = {}

for tp, sub in df.groupby("timepoint"):
    # Lists of tuples
    no_regulation[tp] = list(zip(sub.loc[sub["category"] == "no_regulation", "gene_1"],
                                 sub.loc[sub["category"] == "no_regulation", "gene_2"]))
    potential_regulation[tp] = list(zip(sub.loc[sub["category"] == "potential_regulation", "gene_1"],
                                        sub.loc[sub["category"] == "potential_regulation", "gene_2"]))
    # Pivot to matrix
    corr_mat = sub.pivot_table(index="gene_1", columns="gene_2", values="correlation")
    # make symmetric since we only stored one order per pair
    corr_mat = corr_mat.combine_first(corr_mat.T)
    pairwise_gene_gene_correlation_matrix[tp] = corr_mat

# === Extract t1, t2, t3 structures ===
no_regulation_t1 = no_regulation["t1"]
no_regulation_t2 = no_regulation["t2"]

potential_regulation_t1 = potential_regulation["t1"]
potential_regulation_t2 = potential_regulation["t2"]

pairwise_gene_gene_correlation_matrix_t1 = pairwise_gene_gene_correlation_matrix["t1"]
pairwise_gene_gene_correlation_matrix_t2 = pairwise_gene_gene_correlation_matrix["t2"]

# Optional sanity check
for tp in ["t1", "t2"]:
    print(f"{tp}: {len(no_regulation[tp])} no-reg pairs, "
          f"{len(potential_regulation[tp])} potential-reg pairs, "
          f"matrix {pairwise_gene_gene_correlation_matrix[tp].shape}")


In [ ]:
# --- Step 2: Twin/random correlations at day 2 ---
twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t1 = calculate_twin_random_pair_correlations(
    all_t1_measurements, t1_twins, curr_gene_list
)
# print(twin_pair_correlation_matrix_t2)
if plot_correlation_matrices_as_heatmap:
    plot_matrix_as_heatmap(corr_matrix=twin_pair_correlation_matrix_t1, gene_list=curr_gene_list, no_regulation=no_regulation_t1, potential_regulation=potential_regulation_t1,
        title=f"Twin pair correlations at time d{t1}", add_gene_labels=True, add_time=False, gray_out_no_reg=True, black_out_self=True
    )

    plot_matrix_as_heatmap(corr_matrix=random_pair_correlation_matrix_t1, gene_list=curr_gene_list, no_regulation=no_regulation_t1, potential_regulation=potential_regulation_t1,
        title=f"Random pair correlations across both time points", add_gene_labels=True, add_time=False, time=[t1], gray_out_no_reg=True, black_out_self=True
    )

# --- Step 3: Classify regulation type: single-state vs multiple-states ---
multiple_states_gene_pairs_t1, single_state_regulation_t1 = differentiate_single_state_reg_and_multiple_states(
    all_t1_measurements, potential_regulation_t1, twin_pair_correlation_matrix_t1, random_pair_correlation_matrix_t1, curr_gene_list, z_score_threshold=z_score_threshold_two_states
)
print(multiple_states_gene_pairs_t1, single_state_regulation_t1)

In [ ]:
# --- Step 2: Twin/random correlations at day 4 ---
twin_pair_correlation_matrix_t2, random_pair_correlation_matrix_t2 = calculate_twin_random_pair_correlations(
    all_t2_measurements, t2_twins, curr_gene_list
)
# print(twin_pair_correlation_matrix_t2)
if plot_correlation_matrices_as_heatmap:
    plot_matrix_as_heatmap( corr_matrix=twin_pair_correlation_matrix_t2, gene_list=curr_gene_list, no_regulation=no_regulation_t2, potential_regulation=potential_regulation_t2,
        title=f"Twin pair correlations at time d{t2}", add_gene_labels=True, add_time=False, time=[t2], gray_out_no_reg=True, vmin = -0.4, vmax=0.4, black_out_self=True
    )

    plot_matrix_as_heatmap( corr_matrix=random_pair_correlation_matrix_t2, gene_list=curr_gene_list, no_regulation=no_regulation_t2, potential_regulation=potential_regulation_t2,
        title=f"Random pair correlations across both time points", add_gene_labels=True, add_time=False, time=[t2], gray_out_no_reg=True, vmin = -0.4, vmax=0.4, black_out_self=True
    )

# # --- Step 3: Classify regulation type: single-state vs multiple-states ---
multiple_states_gene_pairs_t2, single_state_regulation_t2 = differentiate_single_state_reg_and_multiple_states(
    all_t2_measurements, potential_regulation_t2, twin_pair_correlation_matrix_t2, random_pair_correlation_matrix_t2, curr_gene_list, z_score_threshold=z_score_threshold_two_states
)
print(multiple_states_gene_pairs_t2, single_state_regulation_t2)

In [ ]:
if len(multiple_states_gene_pairs_t1) > 0:

    multiple_states_no_reg, multiple_states_and_reg = identify_reg_if_multiple_states(
        twin_pair_correlation_matrix_t1,twin_pair_correlation_matrix_t2,random_pair_correlation_matrix_t1,
        random_pair_correlation_matrix_t2,multiple_states_gene_pairs_t1,curr_gene_list
        )
else:
    multiple_states_no_reg, multiple_states_and_reg = [], []

In [63]:
# --- Step 4: Print summary of results ---
all_gene_pairs = list(product(curr_gene_list, repeat=2))
if have_any_output:
    print_summary(no_regulation_t1, single_state_regulation_t1, multiple_states_no_reg, multiple_states_and_reg)

In [64]:
# ----------------------------------
# Collect all classified pairs
# ----------------------------------
scenario_pair_lists = {
    "single-state, no regulation": no_regulation_t2,
    "single-state, regulation": single_state_regulation_t2,
    "multiple states":multiple_states_gene_pairs_t2
}

records = []

for scenario, pairs in scenario_pair_lists.items():
    for g1, g2 in pairs:
        g1, g2 = sorted((g1, g2))   # normalize
        records.append({
            "gene_1": g1,
            "gene_2": g2,
            "scenario": scenario,
            "timepoint": "t2"        # optional but strongly recommended
        })

df_pair_classification = pd.DataFrame(records)

# sanity check
assert not df_pair_classification.duplicated(
    ["gene_1", "gene_2", "timepoint"]
).any()

df_pair_classification.to_csv(f"{output_path}/all_gene_pair_classification_{gene_set_name}_day_5.csv")

In [ ]:
consistent_pairs = (
    set(potential_regulation_t1)
    & set(potential_regulation_t2)
)

consistent_corr = []

for g1, g2 in consistent_pairs:
    # optional: enforce ordering to avoid (A,B) vs (B,A)
    g1, g2 = sorted((g1, g2))

    c1 = pairwise_gene_gene_correlation_matrix_t1.loc[g1, g2]
    c2 = pairwise_gene_gene_correlation_matrix_t2.loc[g1, g2]

    # ignore zero / NaN correlations
    if any(np.isnan([c1, c2])) or any(c == 0 for c in (c1, c2)):
        continue

    if np.sign(c1) == np.sign(c2):
        consistent_corr.append((g1, g2))
consistent_corr = sorted(consistent_corr)
print(len(consistent_corr))
consistent_corr

In [68]:
# --- Step 5: Infer directionality of single-state interactions ---
infer_direction_for_which_edges = "all-potential-regulation"
p_value_threshold_cross_correlation = 0.05
n_cores = 19

# [TODO] Why is it 0 or Nan??

In [ ]:
# --- Step 5: Infer directionality of single-state interactions ---
infer_direction_for_which_edges = "all-potential-regulation"
p_value_threshold_cross_correlation = 0.05
n_cores = 19

if infer_direction_for_which_edges == "single-state" :
    if len(single_state_regulation_t1) > 0:
        bidirectional_pairs = {(a, b) for (a, b) in single_state_regulation_t1} | \
                  {(b, a) for (a, b) in single_state_regulation_t1}
        # Add self-pairs
        genes = {g for pair in single_state_regulation_t1 for g in pair}
        self_pairs = {(g, g) for g in genes}
        # Final
        all_gene_pairs = bidirectional_pairs | self_pairs
        all_gene_pairs = list(all_gene_pairs)
        direction_matrix = get_cross_correlations(across_t_twin1, across_t_twin2, gene_pairs=all_gene_pairs)

        final_directed_edges = identify_actual_directed_edges(across_t_twin1, across_t_twin2, direction_matrix, gene_pairs=all_gene_pairs, threshold = p_value_threshold_cross_correlation, n_cores_to_use = n_cores, verbose = True)

elif infer_direction_for_which_edges == "all-potential-regulation":
        if len(consistent_corr) > 0:
                combined_list = consistent_corr
                bidirectional_pairs = {(a, b) for (a, b) in combined_list} | \
                      {(b, a) for (a, b) in combined_list}
                genes = {g for pair in combined_list for g in pair}
                self_pairs = {(g, g) for g in genes}

                # Final
                all_gene_pairs_all_reg = bidirectional_pairs | self_pairs
                all_gene_pairs_all_reg = list(all_gene_pairs_all_reg)

                direction_matrix = get_cross_correlations(across_t_twin1, across_t_twin2, gene_pairs=all_gene_pairs_all_reg)
                final_directed_edges = identify_actual_directed_edges(across_t_twin1, across_t_twin2, direction_matrix, gene_pairs=all_gene_pairs_all_reg, threshold = p_value_threshold_cross_correlation, n_cores_to_use = n_cores, verbose = True)
        else:
                final_directed_edges = []
                direction_matrix = pd.DataFrame(
                    np.zeros((len(gene_list), len(gene_list))),
                    index=gene_list,
                    columns=gene_list
                )
else:
    print("running all pairs")
    direction_matrix = get_cross_correlations(across_t_twin1, across_t_twin2, gene_pairs=all_gene_pairs)
    final_directed_edges = identify_actual_directed_edges(across_t_twin1, across_t_twin2, direction_matrix, gene_pairs=all_gene_pairs, threshold = p_value_threshold_cross_correlation, n_cores_to_use = n_cores, verbose = True)
print(final_directed_edges)
# print(pre_threshold_direction_matrix)
direction_matrix = direction_matrix.reindex(
index=curr_gene_list,
columns=curr_gene_list,
fill_value=0
)
unfiltered_direction_matrix = direction_matrix
if final_directed_edges:
    for i in direction_matrix.index:
        for j in direction_matrix.columns:
            if i != j and (i, j) not in final_directed_edges:
                direction_matrix.loc[i,j] = 0
if plot_correlation_matrices_as_heatmap and not direction_matrix.empty:
      all_gene_pairs = list(product(curr_gene_list, repeat=2))
      no_reg_pairs = [pair for pair in all_gene_pairs if pair not in final_directed_edges]
      if infer_direction_for_which_edges == "all-potential-regulation" and (multiple_states_gene_pairs_t1+multiple_states_gene_pairs_t2):
          plot_matrix_as_heatmap(
              corr_matrix=direction_matrix,
              gene_list=curr_gene_list,
              no_regulation=no_reg_pairs,
              potential_regulation=final_directed_edges,
              title=r"twin cross-correlation $\hat{\rho}^{\dagger}_{x(t_{1}) \to y(t_{2})}$",
              add_gene_labels=True,
              add_time=False,
              time=[t1, t2],
              gray_out_no_reg=True,
              black_out_self = True,
              symmetric = False,
              draw_diagonal_multi_state_reg = True,
              multi_state_reg_edges = multiple_states_gene_pairs_t1 + multiple_states_gene_pairs_t2
          )
      else:
          plot_matrix_as_heatmap(
              corr_matrix=direction_matrix,
              gene_list=curr_gene_list,
              no_regulation=no_reg_pairs,
              potential_regulation=final_directed_edges,
              title=r"twin cross-correlation $\hat{\rho}^{\dagger}_{x(t_{1}) \to y(t_{2})}$",
              add_gene_labels=True,
              add_time=False,
              time=[t1, t2],
              gray_out_no_reg=True,
              black_out_self = True,
              symmetric = False
            )

In [ ]:
import json
multiple_states = multiple_states_gene_pairs_t1 + multiple_states_gene_pairs_t2
directional_gene_correlation_data = {
    "gene_list": list(curr_gene_list),
    "corr_matrix": direction_matrix.values.tolist(),
    "no_regulation_pairs": [list(p) for p in no_reg_pairs],
    "final_directed_edges": [list(p) for p in final_directed_edges],
    "multi_state_regulation_pairs": [list(p) for p in multiple_states]
}

with open(f"{output_path}/directional_gene_correlation_data.json", "w") as f:
    json.dump(directional_gene_correlation_data, f, indent=2)
